In [3]:
from pathlib import Path
import os

# Move from /notebooks to project root
ROOT = Path(os.getcwd()).parent

# Folder where your PDFs are
PDF_FOLDER = ROOT / "data" / "raw"

# Find ALL pdfs recursively inside /data/raw/
pdf_files = list(PDF_FOLDER.rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found inside {PDF_FOLDER}")

print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(" -", f)


Found 3 PDF files:
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\DZSF_2024_Sensorbasierte_Technologien.pdf
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\jobstfinke_daniel_Guterzuglaengsdynamik.pdf
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\raw\peche_florian_Bremsgestaengestelleruberwachung.pdf


In [89]:
from langchain_community.document_loaders import PyPDFLoader # Langchain PDF Loader change if necessary
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def text_formatter(text: str) -> str:
    """Format text function to clean up extracted text."""

    # Add if neccessary more formatting steps
    # Getting rid of multiple new lines
    text = re.sub(r"\n+", " ", text)
    
    return text.strip()

def read_pdf(pdf_path: str | Path):
    """Read PDF and return list of cleaned pages."""
    pdf_path = Path(pdf_path)
    loader = PyPDFLoader(str(pdf_path))

    raw_docs = loader.load()   # List[Document]

    cleaned_docs = []

    for d in raw_docs:
        page_number = d.metadata.get("page", None)

        cleaned_text = text_formatter(d.page_content)

        new_doc = Document(
            page_content=cleaned_text,
            metadata={
                "source": pdf_path.name,          # filename
                "filepath": str(pdf_path),        # full path
                "page": page_number + 1,              # page number
                #"doctype": "norm",                # optional
                "doc_id": pdf_path.stem.lower(),  # e.g. "din_5566_2"
            }
        )

        cleaned_docs.append(new_doc)

    return cleaned_docs
docs = read_pdf(str(PDF_PATH))
#print(docs[5])  # erstes Document Objekt anzeigen


In [ ]:
#Multi docs use case
all_docs = []
for pdf_path in pdf_files:
    docs = read_pdf(pdf_path)
    all_docs.extend(docs)


In [81]:
#Chunking the documents
def chunk_documents(docs: list[Document], chunk_size: int = 1000, chunk_overlap: int = 200):
    
    """Split cleaned documents into smaller pieces while preserving metadata."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                       #The metadata will be passed with langchain to the chunks 
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(docs)
    return chunks


In [ ]:
#Testing spot for chunking
chunked_docs = chunk_documents(docs)
print(f"Number of chunks created: {len(chunked_docs)}")
print(chunked_docs[8].metadata)

Number of chunks created: 53
{'source': 'din_5566_2.pdf', 'filepath': 'c:\\Users\\maumo\\OneDrive\\Dokumente\\rag_project\\data\\raw\\din_5566_2.pdf', 'page': 2, 'doc_id': 'din_5566_2'}


In [48]:
import re

DIN_HEADER_PATTERNS = [
    r"DEUTSCHE\s*NORM.*",
    r"Alleinverkauf.*Beuth\s+Verlag.*",
    r"Printed\s*copies\s*are\s*uncontrolled.*",
    r"Datum\s*/\s*Uhrzeit\s*des\s*Ausdrucks:.*",
    r"www\.din\.de|www\.beuth\.de",
    r"ICS\s*\d{2}\.\d{3}\.\d{2}",
    r"ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â©.*",
]

def clean_din_text(text: str) -> str:
    # 1) Trennstriche ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼ber Zeilen umbrechen
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # 2) Kopf-/FuÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸zeilen raus
    for pat in DIN_HEADER_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # 3) ZeilenumbrÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼che normalisieren
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)        # Zeilen trimmen
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)        # einzelne \n ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ Leerzeichen
    text = re.sub(r"\n{3,}", "\n\n", text)              # >2 \n ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ 2 \n

    # 4) Mehrere Spaces zusammenfassen
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

print("Seiten vor Cleaning:", len(docs))
print("Erste 500 Zeichen vorher:\n")
print(docs[12].page_content[:500])



Seiten vor Cleaning: 17
Erste 500 Zeichen vorher:

DIN 5566-2:2020-05 
13 
Anhang A 
(normativ) 
 
Sichtbedingungen (schematisch) 
MaÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸e in Millimeter 
 
Legende 
1 Augenposition grÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¶ÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸te Person, stehend, nach DIN 5566-1:2020-05, Tabelle 1 
2 Augenposition grÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¶ÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸te Person, sitzend, nach DIN 5566-1:2020-05, Tabelle 1 
3 Augenposition kleinste Person, sitzend, nach DIN 5566-1:2020-05, Tabelle 1 
4 Sicht auf hohe Signale (grÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¶ÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸te Person, stehend) bei Frontscheibenoberkante 1 950 mm ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼ber 
FuÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸bodenoberkante 
5 Sicht auf hohe Signale (grÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¶ÃƒÆ’Ã†â€™Ãƒâ€¦Ã‚Â¸te Person, sitzend


In [52]:
for d in docs:
    d.page_content = clean_din_text(d.page_content)

print("Erste 500 Zeichen nach Cleaning:\n")
print(docs[4].page_content[:500])


Erste 500 Zeichen nach Cleaning:

DIN 5566-2:2020-05 5 4 PlÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¤tze fÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼r das Personal 4.1 Allgemeines Die FÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼hrerrÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¤ume mÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼ssen so aus gefÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼hrt werden , dass Einmannbedienung sicherges tellt ist und dass der EisenbahnfahrzeugfÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼hrer das Fahrzeug sowohl im Sitzen als auch im Stehen fÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼hren kann. Bei besonderen Einsatzbedingungen kann davon abgewichen werden. Weiterhin muss im FÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¼hrerraum eine Begleitersitzgelegenheit vor gesehen werden, von d er aus die Beobachtung der Strecke mÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¶glich sein muss. 4.2 Anordnung der PlÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¤tze Der Platz des Eisenba


## Proccesing echte Dokumenten

In [37]:
%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd:", cwd)
print("ROOT:", ROOT)


cwd: c:\Users\maumo\OneDrive\Dokumente\rag_project\notebooks
ROOT: c:\Users\maumo\OneDrive\Dokumente\rag_project


In [41]:
from collections import Counter
from datetime import datetime

import src.processing as processing

pdf_path = ROOT / "data" / "raw" / "peche_florian_Bremsgestaengestelleruberwachung.pdf"
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
clean_out = ROOT / "data" / "clean_pages" / f"my_doc_finke{run_id}.jsonl"
chunk_out = ROOT / "data" / "chunks" / f"my_doc_chunks_finke{run_id}.jsonl"

cfg = processing.IngestConfig(
    skip_first_pages=2,
    min_chars=80,
    drop_toc=True,
    drop_noise_pages=True,
    stop_at_references=False,
    repair_encoding=True,
    chunk_size=1000,
    chunk_overlap=200,
    dedupe=True,
    dedupe_min_chars=120,
)

pages, ingest_stats = processing.load_and_clean_pdf(pdf_path, cfg, verbose=True)
chunks, chunk_stats = processing.split_into_chunks(pages, cfg)

processing.save_jsonl(pages, clean_out)
processing.save_jsonl(chunks, chunk_out)

page_lengths = sorted(len(d.page_content) for d in pages)
chunk_lengths = sorted(len(d.page_content) for d in chunks)
section_counts = Counter(d.metadata.get("section", "unknown") for d in pages)
page_counter = Counter(d.metadata.get("page") for d in chunks)

def q(sorted_vals, p):
    if not sorted_vals:
        return 0
    idx = int((len(sorted_vals) - 1) * p)
    return sorted_vals[idx]

joined_pages = "\n".join(d.page_content for d in pages)
mojibake_markers = ["Ã", "â", "Â", "�"]
mojibake_hits = sum(joined_pages.count(m) for m in mojibake_markers)

print("Outputs:")
print(" -", clean_out)
print(" -", chunk_out)

print("\nIngest stats:")
print(ingest_stats)

print("\nChunk stats:")
print(chunk_stats)

print("\nCounts:")
print(" - pages:", len(pages))
print(" - chunks:", len(chunks))
print(" - sections:", dict(section_counts))

print("\nPage length stats (chars):")
print(" - min/p50/p90/max:", q(page_lengths, 0.0), q(page_lengths, 0.5), q(page_lengths, 0.9), q(page_lengths, 1.0))

print("\nChunk length stats (chars):")
print(" - min/p50/p90/max:", q(chunk_lengths, 0.0), q(chunk_lengths, 0.5), q(chunk_lengths, 0.9), q(chunk_lengths, 1.0))

print("\nTop 10 pages by chunk count:")
print(page_counter.most_common(10))

print("\nMojibake marker count (lower is better):", mojibake_hits)

print("\nPreview first cleaned page:")
print(pages[0].metadata if pages else {})
print(pages[0].page_content[:400] if pages else "")


[ingest] peche_florian_Bremsgestaengestelleruberwachung.pdf: kept 103/121 pages (front 2, toc 6, noise 8, short 2)
Outputs:
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\clean_pages\my_doc_finke20260218_120918.jsonl
 - c:\Users\maumo\OneDrive\Dokumente\rag_project\data\chunks\my_doc_chunks_finke20260218_120918.jsonl

Ingest stats:
{'raw_pages': 121, 'kept_pages': 103, 'dropped_front': 2, 'dropped_toc': 6, 'dropped_noise': 8, 'dropped_short': 2, 'references_pages': 4, 'appendix_pages': 0}

Chunk stats:
{'chunks': 222, 'deduped': 0}

Counts:
 - pages: 103
 - chunks: 222
 - sections: {'body': 82, 'references': 21}

Page length stats (chars):
 - min/p50/p90/max: 90 1565 1987 2409

Chunk length stats (chars):
 - min/p50/p90/max: 42 869 984 999

Top 10 pages by chunk count:
[(7, 3), (9, 3), (23, 3), (25, 3), (26, 3), (28, 3), (31, 3), (32, 3), (33, 3), (34, 3)]

Mojibake marker count (lower is better): 0

Preview first cleaned page:
{'source': 'peche_florian_Bremsgestaengestelleruber

In [42]:
from langchain_community.retrievers import BM25Retriever
import src.processing as processing

# Letzten Chunk-Export ausw?hlen (optional manuell setzen)
chunk_files = sorted((ROOT / 'data' / 'chunks').glob('my_doc_chunks*.jsonl'), key=lambda p: p.stat().st_mtime, reverse=True)
if not chunk_files:
    raise FileNotFoundError('Keine Chunk-JSONL unter data/chunks gefunden.')
chunk_path = chunk_files[0]

chunks = processing.load_jsonl(chunk_path)
bm25 = BM25Retriever.from_documents(chunks)

def show_top_chunks(query: str, k: int = 5):
    bm25.k = k
    docs = bm25.invoke(query)
    print(f'Chunk-Datei: {chunk_path}')
    print(f'Query: {query}')
    print(f'Top-{k} Chunks:')
    for i, d in enumerate(docs, start=1):
        meta = d.metadata
        preview = d.page_content[:500].replace('\n', ' ')
        print(f'\n[{i}] page={meta.get("page")}, section={meta.get("section")}, source={meta.get("source")}')
        print(preview)

# Beispiel-Query
show_top_chunks('Welche sensorbasierten Anwendungsfälle werden im Schienenverkehr genannt?', k=5)


Chunk-Datei: c:\Users\maumo\OneDrive\Dokumente\rag_project\data\chunks\my_doc_chunks_finke20260218_120918.jsonl
Query: Welche sensorbasierten Anwendungsfälle werden im Schienenverkehr genannt?
Top-5 Chunks:

[1] page=31, section=body, source=peche_florian_Bremsgestaengestelleruberwachung.pdf
T-Druck einstellt.  2.4  Betriebliche Prüfungen  Bremsproben sind ein zeitaufwendiger Schritt der Zugvorbereitung  und werden im Schienengüterverkehr nach der Zusammenstellung des Güterzuges durchgeführt. Sie dienen der Überprüfung der Funktionsfä­ higkeit der Bremsanlagen aller Wagen im Zug und stellen deren Fahr­

[2] page=39, section=body, source=peche_florian_Bremsgestaengestelleruberwachung.pdf
2.6. Stand der Wissenschaft Abbildung 2.11: Digitale Automatische Kupplung (Typ 4) des Herstellers Voith (2021) Aktoren ausgerüstet werden sollen. Nachfolgend werden die für diese Arbeit relevanten Digitalisierungskomponenten DAK und ABP näher beschrieben. 2.6.1 Digitale automatische Kupplung Nach mehre